In [ ]:
# !pip install -U datasets
# conda install numpy pandas tqdm nltk jupyter
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
# pip install transformers datasets accelerate evaluate sentencepiece rouge_score
# pip install "protobuf>=3.19.6,<6" --upgrade

In [1]:
import gc
import numpy as np
import pandas as pd
import random
from tqdm.auto import tqdm, trange
import re

import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW

from transformers import AutoModelForSeq2SeqLM, NllbTokenizer, AutoTokenizer
from transformers import DataCollatorForSeq2Seq, get_scheduler, get_constant_schedule_with_warmup
from transformers.optimization import Adafactor
from datasets import load_dataset, Dataset

from accelerate import Accelerator
import evaluate

import wandb

In [2]:
import torch

print("PyTorch compiled with CUDA:", torch.version.cuda)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (compiled):", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

PyTorch compiled with CUDA: 12.6
PyTorch version: 2.7.1+cu126
CUDA available: True
CUDA version (compiled): 12.6
cuDNN version: 90501
GPU name: NVIDIA A100-SXM4-40GB


In [3]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
dataset = load_dataset('/project/lt200246-mmacma/Big_seq2seq/data/text-to-gloss_ver5', cache_dir=None)

# Fine_Tune NLLB

#### Tokenizer

In [4]:
model_path = '/project/lt200246-mmacma/Big_seq2seq/model/facebook/nllb-200-3.3B'

In [5]:
tokenizer = NllbTokenizer.from_pretrained(model_path)
print(len(tokenizer))

256204


In [6]:
tokenizer.additional_special_tokens

['ace_Arab',
 'ace_Latn',
 'acm_Arab',
 'acq_Arab',
 'aeb_Arab',
 'afr_Latn',
 'ajp_Arab',
 'aka_Latn',
 'amh_Ethi',
 'apc_Arab',
 'arb_Arab',
 'ars_Arab',
 'ary_Arab',
 'arz_Arab',
 'asm_Beng',
 'ast_Latn',
 'awa_Deva',
 'ayr_Latn',
 'azb_Arab',
 'azj_Latn',
 'bak_Cyrl',
 'bam_Latn',
 'ban_Latn',
 'bel_Cyrl',
 'bem_Latn',
 'ben_Beng',
 'bho_Deva',
 'bjn_Arab',
 'bjn_Latn',
 'bod_Tibt',
 'bos_Latn',
 'bug_Latn',
 'bul_Cyrl',
 'cat_Latn',
 'ceb_Latn',
 'ces_Latn',
 'cjk_Latn',
 'ckb_Arab',
 'crh_Latn',
 'cym_Latn',
 'dan_Latn',
 'deu_Latn',
 'dik_Latn',
 'dyu_Latn',
 'dzo_Tibt',
 'ell_Grek',
 'eng_Latn',
 'epo_Latn',
 'est_Latn',
 'eus_Latn',
 'ewe_Latn',
 'fao_Latn',
 'pes_Arab',
 'fij_Latn',
 'fin_Latn',
 'fon_Latn',
 'fra_Latn',
 'fur_Latn',
 'fuv_Latn',
 'gla_Latn',
 'gle_Latn',
 'glg_Latn',
 'grn_Latn',
 'guj_Gujr',
 'hat_Latn',
 'hau_Latn',
 'heb_Hebr',
 'hin_Deva',
 'hne_Deva',
 'hrv_Latn',
 'hun_Latn',
 'hye_Armn',
 'ibo_Latn',
 'ilo_Latn',
 'ind_Latn',
 'isl_Latn',
 'ita_Latn',

In [7]:
def fix_tokenizer(tokenizer, new_lang='__thai_gloss__'):
    print('tokenizer len before: ', len(tokenizer))
    
    if new_lang not in tokenizer.additional_special_tokens:
        tokenizer.add_special_tokens({'additional_special_tokens': [new_lang]})

    token_id = tokenizer.convert_tokens_to_ids(new_lang)
    print(token_id)
    print('tokenizer len after: ', len(tokenizer))

In [8]:
fix_tokenizer(tokenizer, '__thai_gloss__')

tokenizer len before:  256204
256204
tokenizer len after:  256205


In [9]:
print(tokenizer.convert_ids_to_tokens([256201, 256202, 256203, 256204]))

['zho_Hant', 'zul_Latn', '<mask>', '__thai_gloss__']


In [10]:
input_language = "tha_Thai"
output_language = '__thai_gloss__'

tokenizer.src_lang = input_language
tokenizer.tgt_lang = output_language

In [6]:
#Test
input = tokenizer("และปิดท้ายกันที่กรุงเทพมหานครและปริมณฑล อุณหภูมิต่ำสุด 24 องศา สูงสุด 35 องศา มีฝนฟ้าคะนองร้อยละ 70 ของพื้นที่ค่ะ")
output = tokenizer("#c กรุงเทพมหานคร + จังหวัด + พื้นที่ใกล้เคียง(กรุงเทพมหานครปริมณฑล)|*วันนี้(/ช่วงนี้)|เย็น|อุณหภูมิต่ำ|#c 20+4(24)|ร้อน|อุณหภูมิสูง|แตะถึง|#c 30+5(35)|#c ฝนตก + ในหลายพื้นที่[มีทิศทางประกอบตั้งแต่ฝนตก ใช้สีหน้า 'ปานกลาง' เป็นตัวเชื่อมกับร้อยละของฝนในแต่ละพื้นที่ ใช้ทิศทางการเคลื่อนไหวรอบ ๆ]|*เปอร์เซ็นต์(/ร้อยละ)|70")
print("input part")
print(input)
print(tokenizer.convert_ids_to_tokens(input.input_ids))
print(tokenizer.decode(input.input_ids))
print("-----------------------------------------")
print("output part")
print(output)
print(tokenizer.convert_ids_to_tokens(output.input_ids))
print(tokenizer.decode(output.input_ids))

input part
{'input_ids': [256175, 7499, 159135, 138939, 15923, 2679, 60353, 146921, 24024, 249098, 248647, 248906, 10382, 8500, 22007, 249061, 12496, 248647, 249557, 254078, 248792, 19921, 249240, 249557, 248906, 179868, 80677, 12070, 3258, 40242, 2202, 248059, 2099, 188900, 10986, 227489, 40242, 7102, 248059, 2099, 188900, 25060, 251682, 248439, 137524, 83091, 248439, 2099, 233806, 5223, 7735, 24809, 148233, 2679, 87223, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
['tha_Thai', '▁และ', 'ปิด', 'ท้าย', 'กัน', 'ที่', 'กร', 'ุง', 'เท', 'พ', 'ม', 'ห', 'าน', 'คร', 'และ', 'ป', 'ริ', 'ม', 'ณ', 'ฑ', 'ล', '▁อ', 'ุ', 'ณ', 'ห', 'ภู', 'มิ', 'ต่', 'ํา', 'สุด', '▁24', '▁', 'อง', 'ศา', '▁ส', 'ูง', 'สุด', '▁35', '▁', 'อง', 'ศา', '▁มี', 'ฝ', 'น', 'ฟ้า', 'คะ', 'น', 'อง', 'ร้อย', 'ละ', '▁70', '▁ของ', 'พื้น', 'ที่', 'ค่ะ', '</s>']
tha_Thai และปิดท้ายกันที่กรุงเ

In [6]:
lens = [len(tokenizer.encode(g)) for g in dataset["train"]["text_raw"]]
print(f"Thai Min: {min(lens)}, Avg: {sum(lens)/len(lens):.2f}, Max: {max(lens)}")

lens = [len(tokenizer.encode(g)) for g in dataset["train"]["text_sign"]]
print(f"Sign Min: {min(lens)}, Avg: {sum(lens)/len(lens):.2f}, Max: {max(lens)}")

Thai Min: 10, Avg: 47.65, Max: 137
Sign Min: 9, Avg: 60.87, Max: 202


In [6]:
max_input_length = 200
max_target_length = 300

def preprocess_function(examples):
    model_inputs = tokenizer(examples["text_raw"],
                             max_length=max_input_length,
                             truncation=True)

    target_output = f'{output_language} {examples["text_sign"]}'
    model_outputs = tokenizer(target_output,
                              max_length=max_target_length,
                              truncation=True)

    model_inputs["labels"] = model_outputs["input_ids"][1:]

    return model_inputs

x = preprocess_function(dataset['train'][0])
print(x)
print((tokenizer.decode(x['labels'])))

{'input_ids': [256175, 248059, 207873, 147060, 6587, 251682, 248439, 137524, 83091, 248439, 2099, 233806, 5223, 5851, 24809, 148233, 2679, 25060, 251682, 248439, 110302, 211830, 55509, 90139, 2679, 17242, 249386, 248584, 127285, 110084, 53115, 248964, 248736, 248988, 248836, 248751, 110084, 249041, 2173, 249726, 249540, 27689, 66647, 70274, 166119, 248439, 2099, 12151, 6061, 248644, 248399, 248059, 179868, 248552, 9175, 248992, 149520, 248988, 2019, 14410, 248584, 6061, 7499, 249039, 248992, 75643, 16149, 248844, 10052, 6587, 73273, 36730, 147132, 16086, 175683, 94, 39579, 37482, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [256204, 248059, 207873, 147060, 249335, 249469, 248080, 165864, 248439, 110302, 6921, 43283, 148233, 2679, 249335, 5454, 26558, 66553, 4530, 

In [7]:
tokenized_dataset = dataset.map(preprocess_function)

Map:   0%|          | 0/3086 [00:00<?, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

### Model config

In [15]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_path, device_map='auto')
model.resize_token_embeddings(len(tokenizer))

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

M2M100ScaledWordEmbedding(256205, 2048, padding_idx=1)

In [16]:
model.device

device(type='cuda', index=0)

In [17]:
added_token_id = tokenizer.convert_tokens_to_ids(input_language)
similar_lang_id = tokenizer.convert_tokens_to_ids(output_language)

#Add new language token
model.model.shared.weight.data[added_token_id] = model.model.shared.weight.data[similar_lang_id]

In [18]:
print(similar_lang_id)
print(added_token_id)
print(model.model.shared.weight.data[added_token_id])
print(model.model.shared.weight.data[similar_lang_id])

256204
256175
tensor([ 0.0027,  0.0026,  0.0024,  ..., -0.0133, -0.0311, -0.0459],
       device='cuda:0')
tensor([ 0.0027,  0.0026,  0.0024,  ..., -0.0133, -0.0311, -0.0459],
       device='cuda:0')


In [19]:
print(model.model.shared.weight.data[added_token_id - 5])

tensor([-0.0477, -0.0201,  0.0608,  ..., -0.0699,  0.2759,  0.0598],
       device='cuda:0')


In [20]:
print(model.config)

M2M100Config {
  "activation_dropout": 0.0,
  "activation_function": "relu",
  "architectures": [
    "M2M100ForConditionalGeneration"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": 0,
  "d_model": 2048,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 8192,
  "decoder_layerdrop": 0,
  "decoder_layers": 24,
  "decoder_start_token_id": 2,
  "dropout": 0.1,
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 8192,
  "encoder_layerdrop": 0,
  "encoder_layers": 24,
  "eos_token_id": 2,
  "init_std": 0.02,
  "is_encoder_decoder": true,
  "max_length": 200,
  "max_position_embeddings": 1024,
  "model_type": "m2m_100",
  "num_hidden_layers": 24,
  "pad_token_id": 1,
  "scale_embedding": true,
  "torch_dtype": "float32",
  "transformers_version": "4.53.2",
  "use_cache": true,
  "vocab_size": 256205
}



In [21]:
print(model.generation_config)

GenerationConfig {
  "bos_token_id": 0,
  "decoder_start_token_id": 2,
  "eos_token_id": 2,
  "max_length": 200,
  "pad_token_id": 1
}



#### Setup Dataloader

In [8]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding="longest", return_tensors='pt')

In [9]:
tokenized_dataset = tokenized_dataset.remove_columns(dataset['train'].column_names)
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3086
    })
    eval: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 900
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 900
    })
})

In [10]:
features = [tokenized_dataset['train'][i] for i in range(2)]
data_collator(features)

{'input_ids': tensor([[256175, 248059, 207873, 147060,   6587, 251682, 248439, 137524,  83091,
         248439,   2099, 233806,   5223,   5851,  24809, 148233,   2679,  25060,
         251682, 248439, 110302, 211830,  55509,  90139,   2679,  17242, 249386,
         248584, 127285, 110084,  53115, 248964, 248736, 248988, 248836, 248751,
         110084, 249041,   2173, 249726, 249540,  27689,  66647,  70274, 166119,
         248439,   2099,  12151,   6061, 248644, 248399, 248059, 179868, 248552,
           9175, 248992, 149520, 248988,   2019,  14410, 248584,   6061,   7499,
         249039, 248992,  75643,  16149, 248844,  10052,   6587,  73273,  36730,
         147132,  16086, 175683,     94,  39579,  37482,      2],
        [256175,  95143,   5561, 151005, 251682, 248439, 110302, 211830,  22007,
         251682, 248439,   2679, 110302, 150804,  48931, 232017,  37971,  41134,
          43087,  70198, 248842,  86381, 249643,   4290, 195558,   2173,  13141,
          97101, 249061,   19

In [11]:
batch_size = 8

test_dataloader = DataLoader(tokenized_dataset["test"],
                             collate_fn=data_collator,
                             batch_size=batch_size)

# Trainer

In [26]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

In [27]:
#Optimize
optimizer = AdamW(model.parameters(), lr=2e-5)


wandb.init(
    project="seq2seq-training-dataver5",
    name="nllb-3.3B",
    mode="offline" 
)

In [28]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/project/lt200246-mmacma/Big_seq2seq/trained_model/model_use_data5/nllb",
    save_strategy= "no",
    num_train_epochs=8,
    do_train=True,
    per_device_train_batch_size=4,
    gradient_checkpointing=True,
    gradient_accumulation_steps=8,

    learning_rate=5e-05,
    lr_scheduler_type="linear",

    do_eval=True,
    eval_strategy="epoch",
    per_device_eval_batch_size=4,

    logging_strategy="epoch",
    report_to="wandb"
)

In [29]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["eval"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

In [30]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
`use_cache=True` is incompatible with gradient checkpointing`. Setting `use_cache=False`...


Epoch,Training Loss,Validation Loss
1,1.463300,0.492288
2,0.416000,0.432513
3,0.338100,0.411192
4,0.285000,0.406829
5,0.244100,0.414854
6,0.206800,0.424415
7,0.178400,0.431214
8,0.159100,0.435098


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


TrainOutput(global_step=776, training_loss=0.411341355019009, metrics={'train_runtime': 4475.8266, 'train_samples_per_second': 5.516, 'train_steps_per_second': 0.173, 'total_flos': 2.926503582724915e+16, 'train_loss': 0.411341355019009, 'epoch': 8.0})

In [31]:
trainer.save_model()

/lustrefs/disk/project/lt200246-mmacma/Big_seq2seq/nllb/env_nllb/lib/python3.10/site-packages/transformers/modeling_utils.py:3685: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


# Inference

In [4]:
model = AutoModelForSeq2SeqLM.from_pretrained("/project/lt200246-mmacma/Big_seq2seq/trained_model/model_use_data5/nllb", device_map='auto')
tokenizer = AutoTokenizer.from_pretrained("/project/lt200246-mmacma/Big_seq2seq/trained_model/model_use_data5/nllb")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
input_language = "tha_Thai"
output_language = '__thai_gloss__'

tokenizer.src_lang = input_language
tokenizer.tgt_lang = output_language

In [12]:
text_input = []
gloss_translate = []
answer = []

model.eval()

for batch in tqdm(test_dataloader):
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        input_ids = batch["input_ids"]
        labels = batch["labels"]

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=batch["attention_mask"],
            max_new_tokens=300,
            length_penalty=0.6,
            early_stopping=True,
            num_beams=4,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.5,
        )

    input_ids = input_ids.cpu().numpy()
    outputs = outputs.cpu().numpy()
    labels = labels.cpu().numpy()

    # Replace -100 in labels with pad_token_id before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    translation = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    label = tokenizer.batch_decode(labels, skip_special_tokens=True)
    input = tokenizer.batch_decode(input_ids, skip_special_tokens=True)

    text_input.extend(input)
    gloss_translate.extend(translation)
    answer.extend(label)

result = pd.DataFrame({
    "text": text_input,
    "true_gloss": answer,
    "predicted_gloss": gloss_translate
})

  0%|          | 0/113 [00:00<?, ?it/s]

In [14]:
result

,text,true_gloss,predicted_gloss
0,ทีนี้ไปตรวจสอบสภาพอากาศแบบรายภาคกันค่ะ,ตอนนี้_วันนี้|ดู|อากาศ|กับ_ที่|ภาคเหนือ|ภาคอีส...,เล่า|อากาศ|กับ_ที่|ภาคเหนือ|ภาคอีสาน|ภาคกลาง|ภ...
1,ภาคอีสาน อากาศเย็นกับมีหมอกบางในตอนเช้า และอุณ...,ภาคอีสาน|เย็น|เช้า|ขาว|หมอก|ภาคอีสาน|ร้อน|อุณห...,ภาคอีสาน|เย็น|เช้า|ขาว|หมอก|ตอนนี้_วันนี้|ร้อน...
2,ปิดท้ายกันที่กรุงเทพมหานครและปริมณฑลนะคะ มีฝนฟ...,สุดท้าย|กรุงเทพมหานครและปริมณฑล|#c ฝนตกในหลายพ...,กรุงเทพมหานครและปริมณฑล|#c ฝนตกในหลายพื้นที่|เ...
3,เพราะฉะนั้น สัปดาห์หน้าอากาศก็จะร้อนต่อเนื่อง ...,สัปดาห์|ต่อไป|อากาศ|ร้อน|ต่อเนื่อง|เที่ยง|ข้าง...,สัปดาห์|ต่อไป|อากาศ|ร้อน|ต่อเนื่อง|เที่ยง|เบลอ
4,ส่วนยอดดอยยังคงหนาวถึงหนาวจัด อุณหภูมิต่ําสุด ...,ภูเขา|ยอดดอย|ตรงนี้|หนาว|#c หนาวมาก|ตรงนี้|หนา...,ภูเขา|ยอดเขา|ตรงนี้|หนาว|#c หนาวมาก|อุณหภูมิต่...
...,...,...,...
895,ในส่วนของสภาพอากาศในระยะ 1-2 วันนี้นะคะ ก็มีคํ...,มี|อากาศ|1-2 วัน|นี้[ระบุช่วงวันที่หรือเดือน]|...,ตอนนี้_วันนี้|กับ_ที่|อากาศ|1-2 วัน|นี้[ระบุช่...
896,ขณะที่ภาคกลางอากาศร้อน กับมีฟ้าหลัวในตอนกลางวั...,ภาคกลาง|ร้อน|กับ_ที่|เที่ยง|ร้อน|ฟ้าหลัว|ภาคกล...,ภาคกลาง|ร้อน|เที่ยง|เบลอ|ภาคกลาง|เย็น|อุณหภูมิ...
897,สําหรับสัปดาห์นี้ค่ะ มวลอากาศเย็นที่ปกคลุมประเ...,สัปดาห์นี้|เย็น|ปกคลุม|ไทยตอนบน|เข้ม|อ่อนลง|ลด...,สัปดาห์|นี้[ระบุช่วงวันที่หรือเดือน]|เย็น|ประเ...
898,ส่วนทางภาคใต้ยังเป็นพื้นที่เดียวนะคะ ที่ช่วงนี...,ภาคใต้|ตรงนี้|1|ภาคใต้|ตอนนี้_วันนี้|ระวัง|กับ...,ภาคใต้|ตรงนี้|1|ภาคใต้|ระวัง|#c ฝนตกหนัก|กระทบ...


In [15]:
result.to_csv("/project/lt200246-mmacma/Big_seq2seq/transcript/dataset_ver5/nllb/nllb_dataver5.csv")